# Soberanos Models
## UST Curve Regime Detection | Soberanos CMT & ASW | Event Study

In [ ]:
# =============================================================================
# CELL 1 — Monotone Convex Interpolation Functions
# =============================================================================
# Original functions for Soberanos bond-level data → CMT construction
# + Generic wrapper for interpolating any yield curve (UST, Sobs pre-CMT)
# =============================================================================

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Core Monotone Convex primitives
# ---------------------------------------------------------------------------

def mc_node_forwards(t, f_disc):
    """Compute node forward rates from discrete forwards using MC method."""
    n = len(t)
    f = np.zeros(n)
    for i in range(1, n - 1):
        h_l = t[i] - t[i - 1]
        h_r = t[i + 1] - t[i]
        f[i] = (f_disc[i - 1] * h_r + f_disc[i] * h_l) / (h_l + h_r)
    f[0] = f_disc[0] - 0.5 * (f[1] - f_disc[0])
    f[-1] = f_disc[-1] - 0.5 * (f[-2] - f_disc[-1])
    return np.maximum(f, 1e-6)


def mc_correct_fwd(f0, fd, f1):
    """Ensure non-negative forward rates via MC correction."""
    etas = np.linspace(0.01, 0.99, 300)
    f_cur = (
        f0 * (1 - 4 * etas + 3 * etas**2)
        + fd * 6 * etas * (1 - etas)
        + f1 * etas * (-2 + 3 * etas)
    )
    if f_cur.min() >= 0:
        return f0, fd, f1
    g = (
        f0 * (1 - 4 * etas + 3 * etas**2)
        + f1 * etas * (-2 + 3 * etas)
    )
    w = 6 * etas * (1 - etas)
    mask = w > 1e-12
    fd_new = max(fd, np.max(-g[mask] / w[mask]) + 1e-8)
    return max(f0, 1e-6), fd_new, max(f1, 1e-6)


def mc_yield_at(t_query, t, y_dec, Q, f_disc, f_node):
    """Interpolate yield at arbitrary maturity using Monotone Convex."""
    hit = np.where(np.abs(t - t_query) < 1e-9)[0]
    if len(hit):
        return float(y_dec[hit[0]] * 100)
    if t_query < t[0] or t_query > t[-1]:
        return np.nan
    i = int(np.clip(np.searchsorted(t, t_query, side="right") - 1, 0, len(t) - 2))
    t0, t1 = t[i], t[i + 1]
    h = t1 - t0
    eta = (t_query - t0) / h
    f0, fd, f1 = mc_correct_fwd(f_node[i], f_disc[i], f_node[i + 1])
    Q_t = Q[i] + h * (
        f0 * (eta - 2 * eta**2 + eta**3)
        + fd * (3 * eta**2 - 2 * eta**3)
        + f1 * (-eta**2 + eta**3)
    )
    return float((Q_t / t_query) * 100)


# ---------------------------------------------------------------------------
# Original Soberanos build_cmt (for real bond-level data from dfsobs.csv)
# ---------------------------------------------------------------------------

SOBS_CMT_TENORS = [6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
MIN_BONDS = 3

def build_cmt_from_bonds(df_clean, tenors=None, min_bonds=MIN_BONDS):
    """Build CMT from individual bond data (Fecha, Maturity, FV.Rate).
    This is the ORIGINAL function for real Soberanos data."""
    if tenors is None:
        tenors = SOBS_CMT_TENORS
    records, skipped, first_error = [], 0, None
    df_work = df_clean.copy()
    df_work["Fecha"] = pd.to_datetime(df_work["Fecha"])
    for date, grp in df_work.groupby("Fecha"):
        grp = grp.dropna(subset=["Maturity", "FV.Rate"]).sort_values("Maturity").drop_duplicates("Maturity")
        if len(grp) < min_bonds:
            skipped += 1
            continue
        t_raw = grp["Maturity"].values
        y_raw = grp["FV.Rate"].values
        try:
            idx = np.argsort(t_raw)
            t_s, y_dec = t_raw[idx], y_raw[idx] / 100.0
            Q = y_dec * t_s
            f_disc = np.diff(Q) / np.diff(t_s)
            f_node = mc_node_forwards(t_s, f_disc)
            row = {"Fecha": date}
            t_min, t_max = t_s.min(), t_s.max()
            for tau in tenors:
                if t_min <= tau <= t_max:
                    row[f"{tau}Y"] = round(mc_yield_at(tau, t_s, y_dec, Q, f_disc, f_node), 4)
                else:
                    row[f"{tau}Y"] = np.nan
            records.append(row)
        except Exception as e:
            if first_error is None:
                first_error = str(e)
            skipped += 1
    if not records:
        print(f"ERROR: 0 records. skipped={skipped}. First error: {first_error}")
        return pd.DataFrame()
    df_cmt = pd.DataFrame(records).sort_values("Fecha").reset_index(drop=True)
    df_cmt["Fecha"] = pd.to_datetime(df_cmt["Fecha"])
    print(f"[build_cmt_from_bonds] CMT built: {len(df_cmt):,} dates | skipped: {skipped:,}")
    return df_cmt


# ---------------------------------------------------------------------------
# Generic MC interpolation: from known tenor→yield columns to target tenors
# Works for both UST and Soberanos pre-CMT data
# ---------------------------------------------------------------------------

def mc_interpolate_curve(df_yields, source_tenors, target_tenors):
    """Interpolate a yield curve DataFrame to target tenors via Monotone Convex.
    
    Parameters
    ----------
    df_yields : DataFrame with 'Fecha' index (or column) and tenor columns (e.g. '6Y','10Y')
    source_tenors : list of floats — tenors present in df_yields (years)
    target_tenors : list of floats — tenors to interpolate to (years)
    
    Returns
    -------
    DataFrame with 'Fecha' as index and '{T}Y' columns for each target tenor.
    """
    if "Fecha" in df_yields.columns:
        dates = df_yields["Fecha"]
        vals = df_yields.drop(columns=["Fecha"])
    else:
        dates = df_yields.index
        vals = df_yields

    t = np.array(source_tenors, dtype=float)
    records = []
    skipped = 0
    for i in range(len(vals)):
        y_pct = vals.iloc[i].values.astype(float)
        mask = ~np.isnan(y_pct)
        if mask.sum() < 3:
            skipped += 1
            continue
        t_valid = t[mask]
        y_valid = y_pct[mask]
        idx = np.argsort(t_valid)
        t_s = t_valid[idx]
        y_dec = y_valid[idx] / 100.0
        Q = y_dec * t_s
        f_disc = np.diff(Q) / np.diff(t_s)
        f_node = mc_node_forwards(t_s, f_disc)
        row = {}
        for tau in target_tenors:
            if t_s.min() <= tau <= t_s.max():
                row[f"{int(tau)}Y"] = round(mc_yield_at(tau, t_s, y_dec, Q, f_disc, f_node), 4)
            else:
                row[f"{int(tau)}Y"] = np.nan
        records.append(row)
    
    result = pd.DataFrame(records)
    # Align with valid dates (skip those that had < 3 data points)
    valid_dates = []
    j = 0
    for i in range(len(vals)):
        y_pct = vals.iloc[i].values.astype(float)
        mask = ~np.isnan(y_pct)
        if mask.sum() >= 3:
            valid_dates.append(dates.iloc[i] if hasattr(dates, 'iloc') else dates[i])
    result.insert(0, "Fecha", valid_dates[:len(result)])
    result["Fecha"] = pd.to_datetime(result["Fecha"])
    result = result.set_index("Fecha")
    if skipped:
        print(f"[mc_interpolate_curve] skipped {skipped} rows (< 3 valid points)")
    print(f"[mc_interpolate_curve] Interpolated: {len(result):,} dates, "
          f"tenors {[f'{int(t)}Y' for t in target_tenors]}")
    return result


print("Cell 1 OK — Monotone Convex functions loaded")
print("  build_cmt_from_bonds() — for real Soberanos bond data")
print("  mc_interpolate_curve() — generic MC interpolation for any curve")

In [ ]:
# =============================================================================
# CELL 2 — Configuration & Data Loading
# =============================================================================
# Toggle USE_DUMMY_DATA to switch between dummy CSV and real bond-level data
# USTs always loaded from USTs90s26s.csv (true data)
# =============================================================================

import os
import warnings
warnings.filterwarnings("ignore")

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  TOGGLE: Set False when running on work computer with real dfsobs.csv  │
# └─────────────────────────────────────────────────────────────────────────┘
USE_DUMMY_DATA = True

# ---------------------------------------------------------------------------
# 1) Load UST CMT yields (TRUE DATA — always available)
# ---------------------------------------------------------------------------
UST_CSV = "USTs90s26s.csv"
UST_TENORS_RAW = [1, 2, 3, 5, 7, 10, 20, 30]  # as they appear in the CSV

df_ust_raw = pd.read_csv(UST_CSV)
df_ust_raw.rename(columns={"date": "Fecha"}, inplace=True)
df_ust_raw["Fecha"] = pd.to_datetime(df_ust_raw["Fecha"], format="mixed", dayfirst=False)

# Standardize column names → '1Y', '2Y', etc.
ust_col_map = {}
for c in df_ust_raw.columns:
    if c == "Fecha":
        continue
    num = c.replace(" yr", "").strip()
    ust_col_map[c] = f"{num}Y"
df_ust_raw.rename(columns=ust_col_map, inplace=True)

df_ust = df_ust_raw.set_index("Fecha").sort_index()
df_ust = df_ust.apply(pd.to_numeric, errors="coerce")

print(f"[UST] Loaded {len(df_ust):,} rows | {df_ust.index.min().date()} → {df_ust.index.max().date()}")
print(f"[UST] Columns: {list(df_ust.columns)}")

# ---------------------------------------------------------------------------
# 2) Load Soberanos pre-CMT data (DUMMY or REAL)
# ---------------------------------------------------------------------------
if USE_DUMMY_DATA:
    # ── DUMMY PATH: load dfsobsprecmt.csv (tenors 6Y, 7Y, 10Y, 20Y) ──
    SOBS_PRE_CSV = "dfsobsprecmt.csv"
    df_sobs_pre = pd.read_csv(SOBS_PRE_CSV)
    df_sobs_pre.rename(columns={"date": "Fecha"}, inplace=True)
    df_sobs_pre["Fecha"] = pd.to_datetime(df_sobs_pre["Fecha"], format="mixed", dayfirst=False)
    df_sobs_pre = df_sobs_pre.set_index("Fecha").sort_index()
    df_sobs_pre = df_sobs_pre.apply(pd.to_numeric, errors="coerce")
    SOBS_SOURCE_TENORS = [6, 7, 10, 20]
    print(f"[SOBS dummy] Loaded {len(df_sobs_pre):,} rows | "
          f"{df_sobs_pre.index.min().date()} → {df_sobs_pre.index.max().date()}")
    print(f"[SOBS dummy] Columns: {list(df_sobs_pre.columns)}")
else:
    # ── REAL PATH: load dfsobs.csv and build CMT from bond-level data ──
    # Uncomment and adjust path as needed on work computer
    # SOBS_BOND_CSV = "dfsobs.csv"
    # dfsobs = pd.read_csv(SOBS_BOND_CSV)
    # ... preprocess dfsobs (Fecha, Maturity, FV.Rate, Bid, Offer) ...
    # df_sobs_cmt_from_bonds = build_cmt_from_bonds(dfsobs, tenors=SOBS_CMT_TENORS)
    # df_sobs_cmt_from_bonds = df_sobs_cmt_from_bonds.set_index("Fecha").sort_index()
    print("[SOBS real] ⚠ Set USE_DUMMY_DATA = False and uncomment the real data path above")
    print("            Requires dfsobs.csv with columns: Fecha, Maturity, FV.Rate, Bid, Offer")

print("\nCell 2 OK — Data loaded")

In [ ]:
# =============================================================================
# CELL 3 — UST Spread Computation & Regime Classification
# =============================================================================
# Computes all standard UST curve spreads in bps
# Classifies each day into one of 6 regimes based on realized spread changes
# =============================================================================

import itertools

# ---------------------------------------------------------------------------
# 1) UST yield changes in bps
# ---------------------------------------------------------------------------
df_ust_bps = df_ust * 100  # yields in bps for spread math

# Standard UST spread pairs (short vs long end)
UST_SPREAD_PAIRS = [
    ("2Y", "5Y"),  ("2Y", "10Y"), ("2Y", "30Y"),
    ("5Y", "10Y"), ("5Y", "30Y"), ("10Y", "30Y"),
]

def spread_label(s, l):
    """Create spread label like '2s10s'."""
    return f"{s.replace('Y','')}s{l.replace('Y','')}s"

# Compute spread levels (long - short, in bps)
df_ust_spreads = pd.DataFrame(index=df_ust_bps.index)
for short_t, long_t in UST_SPREAD_PAIRS:
    lbl = spread_label(short_t, long_t)
    df_ust_spreads[lbl] = df_ust_bps[long_t] - df_ust_bps[short_t]

print(f"[UST Spreads] {list(df_ust_spreads.columns)}")
print(f"[UST Spreads] Latest values (bps):")
print(df_ust_spreads.dropna().tail(1).to_string())

# ---------------------------------------------------------------------------
# 2) Regime Classification Function
# ---------------------------------------------------------------------------
REGIME_LABELS = [
    "Bull Steepener",   # rates fall, curve steepens
    "Bear Steepener",   # rates rise, curve steepens
    "Steepener Twist",  # mixed, net steepener
    "Bull Flattener",   # rates fall, curve flattens
    "Bear Flattener",   # rates rise, curve flattens
    "Flattener Twist",  # mixed, net flattener
]

REGIME_COLORS = {
    "Bull Steepener":   "#4FC3F7",  # light blue
    "Bear Steepener":   "#FF9800",  # orange
    "Steepener Twist":  "#4CAF50",  # green
    "Bull Flattener":   "#F44336",  # red
    "Bear Flattener":   "#AB47BC",  # purple
    "Flattener Twist":  "#F48FB1",  # pink
}


def classify_curve_regime(df_yields_bps, short_col, long_col, lookback):
    """Classify curve regime for each date based on realized changes.
    
    Parameters
    ----------
    df_yields_bps : DataFrame with yield columns in bps, DatetimeIndex
    short_col, long_col : str — column names for short/long tenor
    lookback : int — number of business days for realized change window
    
    Returns
    -------
    DataFrame with columns: 'short_chg', 'long_chg', 'spread_chg', 'regime', 'abs_spread'
    """
    short_chg = df_yields_bps[short_col].diff(lookback)
    long_chg = df_yields_bps[long_col].diff(lookback)
    spread_chg = long_chg - short_chg  # positive = steepening

    regime = pd.Series(index=df_yields_bps.index, dtype="object")
    
    # Steepener (spread widens)
    steep = spread_chg > 0
    # Bull (both rates fall or short falls more)
    bull = (short_chg < 0) & (long_chg <= 0)
    # Bear (both rates rise or long rises more)
    bear = (short_chg >= 0) & (long_chg > 0)
    
    regime[steep & bull] = "Bull Steepener"
    regime[steep & bear] = "Bear Steepener"
    regime[steep & ~bull & ~bear] = "Steepener Twist"
    
    # Flattener (spread narrows)
    flat = spread_chg <= 0
    bull_f = (short_chg <= 0) & (long_chg < 0)
    bear_f = (short_chg > 0) & (long_chg >= 0)
    
    regime[flat & bull_f] = "Bull Flattener"
    regime[flat & bear_f] = "Bear Flattener"
    regime[flat & ~bull_f & ~bear_f] = "Flattener Twist"
    
    # Absolute spread level
    abs_spread = df_yields_bps[long_col] - df_yields_bps[short_col]
    
    result = pd.DataFrame({
        "short_chg": short_chg,
        "long_chg": long_chg,
        "spread_chg": spread_chg.abs(),  # absolute for bar height
        "spread_chg_signed": spread_chg,
        "regime": regime,
        "abs_spread": abs_spread,
    }, index=df_yields_bps.index)
    
    return result.dropna(subset=["regime"])


def get_regime_episodes(regime_series):
    """Extract contiguous regime episodes from a regime Series.
    
    Returns list of dicts: {regime, start, end, duration}
    """
    episodes = []
    current = None
    start = None
    for date, reg in regime_series.items():
        if pd.isna(reg):
            continue
        if reg != current:
            if current is not None:
                episodes.append({
                    "regime": current, "start": start,
                    "end": prev_date, "duration": day_count
                })
            current = reg
            start = date
            day_count = 1
        else:
            day_count += 1
        prev_date = date
    if current is not None:
        episodes.append({
            "regime": current, "start": start,
            "end": prev_date, "duration": day_count
        })
    return pd.DataFrame(episodes)


print("\nCell 3 OK — Regime classification ready")
print(f"  Regimes: {REGIME_LABELS}")
print(f"  UST Spreads: {list(df_ust_spreads.columns)}")

In [ ]:
# =============================================================================
# CELL 4 — UST Curve Regime Panel (Interactive, All Spreads)
# =============================================================================
# Stacked subplots for all 6 UST spreads with regime-colored bars
# + spread level overlay line. Discrete date X-axis (no gaps).
# =============================================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

BG_COLOR = "#000000"
GRID_COLOR = "#333333"
TEXT_COLOR = "#CCCCCC"
TITLE_COLOR = "#FF9800"

def plot_ust_regime_panel(lookback=20, weeks=52, country="US"):
    """Build the full Curve Regime Panel with 6 stacked spread subplots."""
    
    n_days = weeks * 5  # approximate business days
    
    fig = make_subplots(
        rows=6, cols=1, shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=[spread_label(s, l) for s, l in UST_SPREAD_PAIRS],
    )
    
    for idx, (short_t, long_t) in enumerate(UST_SPREAD_PAIRS):
        row = idx + 1
        lbl = spread_label(short_t, long_t)
        
        # Classify regime for this spread
        regime_df = classify_curve_regime(df_ust_bps, short_t, long_t, lookback)
        regime_df = regime_df.tail(n_days)
        
        if regime_df.empty:
            continue
        
        # Use string dates for discrete x-axis (no weekend gaps)
        date_strings = regime_df.index.strftime("%Y-%m-%d").tolist()
        
        # Regime-colored bars (absolute spread change magnitude)
        for regime_name in REGIME_LABELS:
            mask = regime_df["regime"] == regime_name
            if not mask.any():
                continue
            y_vals = regime_df.loc[mask, "spread_chg"].values
            x_vals = [date_strings[i] for i, m in enumerate(mask.values) if m]
            fig.add_trace(go.Bar(
                x=x_vals, y=y_vals,
                marker_color=REGIME_COLORS[regime_name],
                name=regime_name,
                showlegend=(row == 1),
                legendgroup=regime_name,
                hovertemplate="%{x}<br>" + regime_name + "<br>Δ spread: %{y:.1f} bps<extra></extra>",
            ), row=row, col=1)
        
        # Spread level overlay (orange line)
        fig.add_trace(go.Scatter(
            x=date_strings,
            y=regime_df["abs_spread"].values,
            mode="lines",
            line=dict(color="#FF9800", width=1.5),
            name=f"{lbl} Curve" if row == 1 else None,
            showlegend=(row == 1),
            legendgroup="curve_level",
            yaxis=f"y{row * 2}" if row > 1 else "y2",
            hovertemplate="%{x}<br>Spread: %{y:.1f} bps<extra></extra>",
        ), row=row, col=1)
        
        # Y-axis label
        fig.update_yaxes(
            title_text="bps", row=row, col=1,
            gridcolor=GRID_COLOR, zerolinecolor=GRID_COLOR,
            title_font=dict(size=10, color=TEXT_COLOR),
            tickfont=dict(size=9, color=TEXT_COLOR),
        )
    
    # Reduce x-axis tick density — show ~20 evenly spaced labels
    all_dates = classify_curve_regime(df_ust_bps, "2Y", "10Y", lookback).tail(n_days)
    if not all_dates.empty:
        date_strings_all = all_dates.index.strftime("%Y-%m-%d").tolist()
        step = max(1, len(date_strings_all) // 20)
        tick_vals = date_strings_all[::step]
        tick_text = [d[5:] for d in tick_vals]  # show MM-DD only
        fig.update_xaxes(
            row=6, col=1,
            tickvals=tick_vals, ticktext=tick_text,
            tickangle=45, tickfont=dict(size=8, color=TEXT_COLOR),
        )
    
    fig.update_layout(
        height=1200, width=900,
        barmode="overlay",
        plot_bgcolor=BG_COLOR,
        paper_bgcolor=BG_COLOR,
        font=dict(color=TEXT_COLOR, size=10),
        title=dict(
            text=f"CURVE REGIME PANEL — {country}",
            font=dict(color=TITLE_COLOR, size=14, family="Arial Black"),
            x=0.02, y=0.99,
        ),
        legend=dict(
            x=1.02, y=1, font=dict(size=9, color=TEXT_COLOR),
            bgcolor="rgba(0,0,0,0.5)", bordercolor=GRID_COLOR,
        ),
        margin=dict(l=50, r=120, t=40, b=60),
    )
    
    # Hide intermediate x-axes
    for i in range(1, 6):
        fig.update_xaxes(showticklabels=False, row=i, col=1)
    
    # Style subplot titles
    for ann in fig.layout.annotations:
        ann.font = dict(size=11, color="#FFFFFF")
    
    return fig


# ---------------------------------------------------------------------------
# Interactive widget controls
# ---------------------------------------------------------------------------
w_country = widgets.Dropdown(options=["US"], value="US", description="Country:")
w_weeks = widgets.IntText(value=52, description="Weeks:", layout=widgets.Layout(width="150px"))
w_lookback = widgets.IntText(value=20, description="Lookback:", layout=widgets.Layout(width="150px"))
w_btn = widgets.Button(description="Update", button_style="info")
out_regime_panel = widgets.Output()

def _on_update_regime_panel(b):
    with out_regime_panel:
        clear_output(wait=True)
        fig = plot_ust_regime_panel(
            lookback=w_lookback.value,
            weeks=w_weeks.value,
            country=w_country.value,
        )
        fig.show()

w_btn.on_click(_on_update_regime_panel)

display(widgets.HBox([w_country, w_weeks, w_lookback, w_btn]))
display(out_regime_panel)

# Auto-render on load
_on_update_regime_panel(None)

In [ ]:
# =============================================================================
# CELL 5 — Soberanos CMT Construction
# =============================================================================
# Interpolates pre-CMT data [6Y, 7Y, 10Y, 20Y] → full CMT [6Y–15Y]
# Or uses real bond-level data via build_cmt_from_bonds()
# =============================================================================

SOBS_CMT_TARGET = [6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

if USE_DUMMY_DATA:
    # Interpolate dummy pre-CMT: [6, 7, 10, 20] → [6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    df_sobs_cmt = mc_interpolate_curve(
        df_sobs_pre, 
        source_tenors=SOBS_SOURCE_TENORS,  # [6, 7, 10, 20]
        target_tenors=SOBS_CMT_TARGET,
    )
else:
    # REAL PATH — uncomment when running on work computer:
    # df_sobs_cmt = df_sobs_cmt_from_bonds  # from Cell 2 real path
    print("[SOBS CMT] ⚠ Real path not active — set USE_DUMMY_DATA = False in Cell 2")

# Convert to bps for spread computations
df_sobs_bps = df_sobs_cmt * 100

# ---------------------------------------------------------------------------
# Clean summary display
# ---------------------------------------------------------------------------
print(f"{'='*65}")
print(f"  SOBERANOS CMT SUMMARY")
print(f"{'='*65}")
print(f"  Dates : {df_sobs_cmt.index.min().date()} → {df_sobs_cmt.index.max().date()}")
print(f"  Rows  : {len(df_sobs_cmt):,}")
print(f"  Tenors: {list(df_sobs_cmt.columns)}")
print(f"{'─'*65}")
print(f"  Latest yields (%):")
latest = df_sobs_cmt.dropna().tail(1)
for col in latest.columns:
    print(f"    {col}: {latest[col].values[0]:.2f}%")
print(f"{'─'*65}")
print(f"  NaN count per tenor:")
nan_counts = df_sobs_cmt.isna().sum()
for col in nan_counts.index:
    ct = nan_counts[col]
    if ct > 0:
        print(f"    {col}: {ct:,} missing")
    else:
        print(f"    {col}: ✓ complete")
print(f"{'='*65}")

# Quick sanity: show last 5 rows
print("\n  Last 5 dates:")
print(df_sobs_cmt.tail(5).to_string(float_format="%.2f"))
print(f"\nCell 5 OK — df_sobs_cmt ready ({len(df_sobs_cmt):,} dates × {len(df_sobs_cmt.columns)} tenors)")

In [ ]:
# =============================================================================
# CELL 6 — UST Interpolation to Sobs Tenors & ASW Spreads
# =============================================================================
# Interpolates UST curve [1,2,3,5,7,10,20,30]Y → [6–15]Y via Monotone Convex
# Then ASW = Soberano CMT − UST CMT (at matched tenors), in bps
# =============================================================================

# Interpolate UST yields to match Soberanos tenors
df_ust_at_sobs = mc_interpolate_curve(
    df_ust,
    source_tenors=UST_TENORS_RAW,       # [1, 2, 3, 5, 7, 10, 20, 30]
    target_tenors=SOBS_CMT_TARGET,       # [6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
)

# Convert to bps
df_ust_at_sobs_bps = df_ust_at_sobs * 100

# ---------------------------------------------------------------------------
# ASW = Soberano CMT − UST CMT (at matched tenors)
# ---------------------------------------------------------------------------
# Align dates (inner join)
common_dates = df_sobs_bps.index.intersection(df_ust_at_sobs_bps.index)
common_cols = [c for c in df_sobs_bps.columns if c in df_ust_at_sobs_bps.columns]

df_asw = df_sobs_bps.loc[common_dates, common_cols] - df_ust_at_sobs_bps.loc[common_dates, common_cols]
df_asw = df_asw.sort_index()

# Rename columns to indicate ASW
df_asw.columns = [f"{c}_ASW" for c in df_asw.columns]

print(f"{'='*65}")
print(f"  ASW SPREAD SUMMARY (Soberano − UST, bps)")
print(f"{'='*65}")
print(f"  Dates : {df_asw.index.min().date()} → {df_asw.index.max().date()}")
print(f"  Rows  : {len(df_asw):,}")
print(f"  Tenors: {list(df_asw.columns)}")
print(f"{'─'*65}")
print(f"  Latest ASW (bps):")
latest_asw = df_asw.dropna().tail(1)
for col in latest_asw.columns:
    print(f"    {col}: {latest_asw[col].values[0]:.1f} bps")
print(f"{'='*65}")

print(f"\nCell 6 OK — df_asw ready ({len(df_asw):,} dates × {len(df_asw.columns)} tenors)")

In [ ]:
# =============================================================================
# CELL 7 — All Soberanos Inter-Node Spreads + Sector Composites
# =============================================================================
# Builds df_sobs_spreads: all pairwise CMT spreads (long − short) in bps
# Plus belly vs 10s sector and belly vs long composites
# =============================================================================

# ---------------------------------------------------------------------------
# 1) All pairwise spreads between Soberanos CMT nodes
# ---------------------------------------------------------------------------
sobs_tenor_cols = [f"{t}Y" for t in SOBS_CMT_TARGET]

df_sobs_spreads = pd.DataFrame(index=df_sobs_bps.index)

for i, short_c in enumerate(sobs_tenor_cols):
    for long_c in sobs_tenor_cols[i+1:]:
        lbl = spread_label(short_c, long_c)
        df_sobs_spreads[lbl] = df_sobs_bps[long_c] - df_sobs_bps[short_c]

print(f"[SOBS Spreads] {len(df_sobs_spreads.columns)} pairwise spreads computed")

# ---------------------------------------------------------------------------
# 2) Key monitored spreads
# ---------------------------------------------------------------------------
SOBS_KEY_SPREADS = ["6s10s", "6s15s", "7s15s", "6s12s"]
print(f"[SOBS Key Spreads] {SOBS_KEY_SPREADS}")
for sp in SOBS_KEY_SPREADS:
    latest_val = df_sobs_spreads[sp].dropna().iloc[-1]
    print(f"  {sp}: {latest_val:.1f} bps")

# ---------------------------------------------------------------------------
# 3) Sector composites (belly vs 10s sector, belly vs long)
# ---------------------------------------------------------------------------
# Belly: avg of 6Y, 7Y, 8Y, 9Y
# 10s Sector: avg of 10Y, 11Y, 12Y
# Long: avg of 13Y, 14Y, 15Y

belly_cols = ["6Y", "7Y", "8Y", "9Y"]
sector_10s_cols = ["10Y", "11Y", "12Y"]
long_cols = ["13Y", "14Y", "15Y"]

df_sobs_bps_belly = df_sobs_bps[belly_cols].mean(axis=1)
df_sobs_bps_10s = df_sobs_bps[sector_10s_cols].mean(axis=1)
df_sobs_bps_long = df_sobs_bps[long_cols].mean(axis=1)

df_sobs_spreads["belly_vs_10s"] = df_sobs_bps_10s - df_sobs_bps_belly
df_sobs_spreads["belly_vs_long"] = df_sobs_bps_long - df_sobs_bps_belly

print(f"\n[Sector Composites]")
print(f"  belly (avg 6-9Y) vs 10s sector (avg 10-12Y): "
      f"{df_sobs_spreads['belly_vs_10s'].dropna().iloc[-1]:.1f} bps")
print(f"  belly (avg 6-9Y) vs long (avg 13-15Y): "
      f"{df_sobs_spreads['belly_vs_long'].dropna().iloc[-1]:.1f} bps")

# ---------------------------------------------------------------------------
# 4) Same for ASW spreads
# ---------------------------------------------------------------------------
asw_cols_clean = {c: c.replace("_ASW", "") for c in df_asw.columns}
df_asw_clean = df_asw.rename(columns=asw_cols_clean)  # temp rename for spread calc

df_asw_spreads = pd.DataFrame(index=df_asw.index)

asw_tenor_cols = [f"{t}Y" for t in SOBS_CMT_TARGET if f"{t}Y" in df_asw_clean.columns]
for i, short_c in enumerate(asw_tenor_cols):
    for long_c in asw_tenor_cols[i+1:]:
        lbl = spread_label(short_c, long_c) + "_asw"
        df_asw_spreads[lbl] = df_asw_clean[long_c] - df_asw_clean[short_c]

# ASW sector composites
asw_belly = df_asw_clean[[c for c in belly_cols if c in df_asw_clean.columns]].mean(axis=1)
asw_10s = df_asw_clean[[c for c in sector_10s_cols if c in df_asw_clean.columns]].mean(axis=1)
asw_long = df_asw_clean[[c for c in long_cols if c in df_asw_clean.columns]].mean(axis=1)

df_asw_spreads["belly_vs_10s_asw"] = asw_10s - asw_belly
df_asw_spreads["belly_vs_long_asw"] = asw_long - asw_belly

print(f"\n[ASW Spreads] {len(df_asw_spreads.columns)} columns computed")
print(f"\nCell 7 OK — df_sobs_spreads ({len(df_sobs_spreads.columns)} cols) & "
      f"df_asw_spreads ({len(df_asw_spreads.columns)} cols) ready")

In [ ]:
# =============================================================================
# CELL 8 — Soberanos Regime Detection (Absolute Yield & ASW)
# =============================================================================
# Regime classification on Soberanos curve spreads:
#   - Absolute yield: 6s10s, 6s15s, 7s15s, 6s12s, belly_vs_10s, belly_vs_long
#   - ASW level: same spreads but on ASW data
# Stacked panel like Cell 4 but for Soberanos
# =============================================================================

SOBS_REGIME_SPREADS = [
    ("6s10s", "6Y", "10Y"),
    ("6s15s", "6Y", "15Y"),
    ("7s15s", "7Y", "15Y"),
    ("6s12s", "6Y", "12Y"),
    ("belly_vs_10s", None, None),  # composite — handled separately
    ("belly_vs_long", None, None),
]


def classify_sobs_regime(spread_label_name, lookback, use_asw=False):
    """Classify regime for a Soberanos spread (absolute or ASW).
    
    For simple spreads: uses the two tenor columns directly.
    For composites (belly_vs_*): uses pre-computed series.
    """
    if use_asw:
        df_base = df_asw_clean  # ASW levels per tenor (bps)
        df_spread_src = df_asw_spreads
    else:
        df_base = df_sobs_bps   # absolute yields (bps)
        df_spread_src = df_sobs_spreads
    
    # Find the matching entry
    entry = None
    for lbl, s, l in SOBS_REGIME_SPREADS:
        if lbl == spread_label_name:
            entry = (lbl, s, l)
            break
    if entry is None:
        raise ValueError(f"Unknown spread: {spread_label_name}")
    
    lbl, short_col, long_col = entry
    suffix = "_asw" if use_asw else ""
    col_name = lbl + suffix if use_asw else lbl
    
    if short_col is not None and long_col is not None:
        # Simple two-tenor spread
        return classify_curve_regime(df_base, short_col, long_col, lookback)
    else:
        # Composite spread — need to construct synthetic "short" and "long"
        if "belly_vs_10s" in lbl:
            if use_asw:
                short_series = df_asw_clean[[c for c in belly_cols if c in df_asw_clean.columns]].mean(axis=1)
                long_series = df_asw_clean[[c for c in sector_10s_cols if c in df_asw_clean.columns]].mean(axis=1)
            else:
                short_series = df_sobs_bps[belly_cols].mean(axis=1)
                long_series = df_sobs_bps[sector_10s_cols].mean(axis=1)
        elif "belly_vs_long" in lbl:
            if use_asw:
                short_series = df_asw_clean[[c for c in belly_cols if c in df_asw_clean.columns]].mean(axis=1)
                long_series = df_asw_clean[[c for c in long_cols if c in df_asw_clean.columns]].mean(axis=1)
            else:
                short_series = df_sobs_bps[belly_cols].mean(axis=1)
                long_series = df_sobs_bps[long_cols].mean(axis=1)
        else:
            raise ValueError(f"Unhandled composite: {lbl}")
        
        # Build a temp df for classify_curve_regime
        df_tmp = pd.DataFrame({"_short": short_series, "_long": long_series})
        return classify_curve_regime(df_tmp, "_short", "_long", lookback)


def plot_sobs_regime_panel(lookback=20, weeks=52, use_asw=False):
    """Stacked regime panel for Soberanos spreads."""
    n_days = weeks * 5
    mode_label = "ASW" if use_asw else "Yield"
    
    spread_names = [lbl for lbl, _, _ in SOBS_REGIME_SPREADS]
    
    fig = make_subplots(
        rows=6, cols=1, shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=[f"{s} ({mode_label})" for s in spread_names],
    )
    
    for idx, (spread_name, _, _) in enumerate(SOBS_REGIME_SPREADS):
        row = idx + 1
        regime_df = classify_sobs_regime(spread_name, lookback, use_asw=use_asw)
        regime_df = regime_df.tail(n_days)
        if regime_df.empty:
            continue
        
        date_strings = regime_df.index.strftime("%Y-%m-%d").tolist()
        
        for regime_name in REGIME_LABELS:
            mask = regime_df["regime"] == regime_name
            if not mask.any():
                continue
            y_vals = regime_df.loc[mask, "spread_chg"].values
            x_vals = [date_strings[i] for i, m in enumerate(mask.values) if m]
            fig.add_trace(go.Bar(
                x=x_vals, y=y_vals,
                marker_color=REGIME_COLORS[regime_name],
                name=regime_name,
                showlegend=(row == 1),
                legendgroup=regime_name,
                hovertemplate="%{x}<br>" + regime_name + "<br>Δ: %{y:.1f} bps<extra></extra>",
            ), row=row, col=1)
        
        # Spread level line
        fig.add_trace(go.Scatter(
            x=date_strings,
            y=regime_df["abs_spread"].values,
            mode="lines",
            line=dict(color="#FF9800", width=1.5),
            name="Spread Level" if row == 1 else None,
            showlegend=(row == 1),
            legendgroup="spread_lvl",
            hovertemplate="%{x}<br>Spread: %{y:.1f} bps<extra></extra>",
        ), row=row, col=1)
        
        fig.update_yaxes(
            title_text="bps", row=row, col=1,
            gridcolor=GRID_COLOR, zerolinecolor=GRID_COLOR,
            title_font=dict(size=10, color=TEXT_COLOR),
            tickfont=dict(size=9, color=TEXT_COLOR),
        )
    
    # X-axis ticks on bottom
    regime_df_tick = classify_sobs_regime("6s10s", lookback, use_asw).tail(n_days)
    if not regime_df_tick.empty:
        ds = regime_df_tick.index.strftime("%Y-%m-%d").tolist()
        step = max(1, len(ds) // 20)
        fig.update_xaxes(
            row=6, col=1, tickvals=ds[::step],
            ticktext=[d[5:] for d in ds[::step]],
            tickangle=45, tickfont=dict(size=8, color=TEXT_COLOR),
        )
    for i in range(1, 6):
        fig.update_xaxes(showticklabels=False, row=i, col=1)
    
    fig.update_layout(
        height=1200, width=900, barmode="overlay",
        plot_bgcolor=BG_COLOR, paper_bgcolor=BG_COLOR,
        font=dict(color=TEXT_COLOR, size=10),
        title=dict(
            text=f"SOBERANO CURVE REGIME PANEL — {mode_label}",
            font=dict(color=TITLE_COLOR, size=14, family="Arial Black"),
            x=0.02, y=0.99,
        ),
        legend=dict(
            x=1.02, y=1, font=dict(size=9, color=TEXT_COLOR),
            bgcolor="rgba(0,0,0,0.5)", bordercolor=GRID_COLOR,
        ),
        margin=dict(l=50, r=120, t=40, b=60),
    )
    for ann in fig.layout.annotations:
        ann.font = dict(size=11, color="#FFFFFF")
    return fig


# ---------------------------------------------------------------------------
# Widget for Soberanos Regime Panel
# ---------------------------------------------------------------------------
w_sobs_mode = widgets.Dropdown(options=["Yield", "ASW"], value="Yield", description="Mode:")
w_sobs_lb = widgets.IntText(value=20, description="Lookback:", layout=widgets.Layout(width="150px"))
w_sobs_wk = widgets.IntText(value=52, description="Weeks:", layout=widgets.Layout(width="150px"))
w_sobs_btn = widgets.Button(description="Update", button_style="info")
out_sobs_regime = widgets.Output()

def _on_update_sobs_regime(b):
    with out_sobs_regime:
        clear_output(wait=True)
        fig = plot_sobs_regime_panel(
            lookback=w_sobs_lb.value,
            weeks=w_sobs_wk.value,
            use_asw=(w_sobs_mode.value == "ASW"),
        )
        fig.show()

w_sobs_btn.on_click(_on_update_sobs_regime)
display(widgets.HBox([w_sobs_mode, w_sobs_lb, w_sobs_wk, w_sobs_btn]))
display(out_sobs_regime)
_on_update_sobs_regime(None)

In [ ]:
# =============================================================================
# CELL 9 — Soberano Evolution Widget (Line Only, No Bars)
# =============================================================================
# Product: [CMT Tenor, Spread to UST]
# Type: [Level (1 box), Spread (2 boxes), Butterfly (3 boxes)]
# Lookback: [Intraday (1d), 5d, 20d]
# Date range selector
# Shows evolution line ONLY (no delta-change bars)
# =============================================================================


def get_sobs_series(product, stype, t1, t2=None, t3=None):
    """Build the time series for the selected product/type combination.
    
    Returns (series_bps, label) where series is in bps.
    """
    if product == "CMT Tenor":
        base_df = df_sobs_bps
        suffix = ""
    else:  # Spread to UST
        base_df = df_asw.rename(columns={c: c.replace("_ASW", "") for c in df_asw.columns})
        suffix = " ASW"
    
    if stype == "Level":
        col = f"{t1}Y"
        if col not in base_df.columns:
            return None, f"{col} not found"
        return base_df[col], f"{t1}Y{suffix}"
    
    elif stype == "Spread":
        c1, c2 = f"{t1}Y", f"{t2}Y"
        if c1 not in base_df.columns or c2 not in base_df.columns:
            return None, f"{c1} or {c2} not found"
        return base_df[c2] - base_df[c1], f"{t1}s{t2}s{suffix}"
    
    elif stype == "Butterfly":
        c1, c2, c3 = f"{t1}Y", f"{t2}Y", f"{t3}Y"
        for c in [c1, c2, c3]:
            if c not in base_df.columns:
                return None, f"{c} not found"
        # Butterfly = 2*belly - wing1 - wing2
        return 2 * base_df[c2] - base_df[c1] - base_df[c3], f"{t1}s{t2}s{t3}s BF{suffix}"
    
    return None, "Invalid type"


def plot_sobs_evolution(product, stype, t1, t2, t3, lookback, start_dt, end_dt):
    """Plot Soberano evolution: level line with lookback-period changes."""
    
    series, label = get_sobs_series(product, stype, t1, t2, t3)
    if series is None:
        print(f"Error: {label}")
        return
    
    # Filter date range
    series = series.loc[start_dt:end_dt].dropna()
    if len(series) < 2:
        print("Not enough data in selected range")
        return
    
    # Compute changes based on lookback
    lb_map = {"Intraday (1d)": 1, "5d": 5, "20d": 20}
    lb = lb_map.get(lookback, 1)
    changes = series.diff(lb)
    
    # Latest info
    latest_date = series.index[-1]
    latest_level = series.iloc[-1]
    latest_chg = changes.iloc[-1] if not pd.isna(changes.iloc[-1]) else 0
    
    # Use discrete dates (string x-axis)
    date_strings = series.index.strftime("%d-%b-%y").tolist()
    
    fig = go.Figure()
    
    # Level line (primary — cyan)
    fig.add_trace(go.Scatter(
        x=date_strings, y=series.values,
        mode="lines+markers",
        line=dict(color="#00E5FF", width=2),
        marker=dict(size=3, color="#00E5FF"),
        name=f"{label} Level",
        hovertemplate="%{x}<br>Level: %{y:.1f} bps<extra></extra>",
    ))
    
    fig.update_layout(
        height=450, width=950,
        plot_bgcolor=BG_COLOR, paper_bgcolor=BG_COLOR,
        font=dict(color=TEXT_COLOR, size=10),
        title=dict(
            text=(f"<b>Soberano Evolution: {label}</b><br>"
                  f"<span style='font-size:11px;color:#00E5FF'>"
                  f"As of {latest_date.strftime('%Y-%m-%d')} | Level: {latest_level:.1f} bps | "
                  f"{lookback} chg: {latest_chg:+.1f} bps</span>"),
            font=dict(size=13, color="#FFFFFF"),
            x=0.02, y=0.95,
        ),
        yaxis=dict(
            title="Level (bps)", gridcolor=GRID_COLOR, zerolinecolor=GRID_COLOR,
            titlefont=dict(color=TEXT_COLOR), tickfont=dict(color=TEXT_COLOR),
        ),
        xaxis=dict(
            tickangle=45, tickfont=dict(size=8, color=TEXT_COLOR),
        ),
        margin=dict(l=60, r=30, t=70, b=60),
        showlegend=False,
    )
    
    # Thin x-axis ticks — show ~25 labels
    step = max(1, len(date_strings) // 25)
    fig.update_xaxes(tickvals=date_strings[::step], ticktext=date_strings[::step])
    
    fig.show()


# ---------------------------------------------------------------------------
# Widget controls
# ---------------------------------------------------------------------------
w_ev_product = widgets.Dropdown(
    options=["CMT Tenor", "Spread to UST"], value="Spread to UST",
    description="Product:", layout=widgets.Layout(width="200px"))
w_ev_type = widgets.Dropdown(
    options=["Level", "Spread", "Butterfly"], value="Level",
    description="Type:", layout=widgets.Layout(width="160px"))

w_ev_t1 = widgets.Dropdown(options=[6,7,8,9,10,11,12,13,14,15], value=10, description="T1:")
w_ev_t2 = widgets.Dropdown(options=[6,7,8,9,10,11,12,13,14,15], value=15, description="T2:")
w_ev_t3 = widgets.Dropdown(options=[6,7,8,9,10,11,12,13,14,15], value=15, description="T3:")
w_ev_lb = widgets.Dropdown(
    options=["Intraday (1d)", "5d", "20d"], value="Intraday (1d)",
    description="Lookback:", layout=widgets.Layout(width="180px"))

# Date range
min_date = df_sobs_cmt.index.min().date()
max_date = df_sobs_cmt.index.max().date()
w_ev_start = widgets.DatePicker(
    value=pd.Timestamp(max_date) - pd.DateOffset(months=2),
    description="Start:", layout=widgets.Layout(width="200px"))
w_ev_end = widgets.DatePicker(
    value=max_date, description="End:", layout=widgets.Layout(width="200px"))

w_ev_btn = widgets.Button(description="Update", button_style="info")
out_evolution = widgets.Output()

# Show/hide T2, T3 based on type
def _update_type_visibility(change):
    stype = w_ev_type.value
    w_ev_t2.layout.visibility = "visible" if stype in ["Spread", "Butterfly"] else "hidden"
    w_ev_t3.layout.visibility = "visible" if stype == "Butterfly" else "hidden"

w_ev_type.observe(_update_type_visibility, names="value")
_update_type_visibility(None)

def _on_update_evolution(b):
    with out_evolution:
        clear_output(wait=True)
        plot_sobs_evolution(
            product=w_ev_product.value,
            stype=w_ev_type.value,
            t1=w_ev_t1.value,
            t2=w_ev_t2.value,
            t3=w_ev_t3.value,
            lookback=w_ev_lb.value,
            start_dt=str(w_ev_start.value),
            end_dt=str(w_ev_end.value),
        )

w_ev_btn.on_click(_on_update_evolution)

display(widgets.HBox([w_ev_product, w_ev_type, w_ev_t1, w_ev_t2, w_ev_t3, w_ev_lb]))
display(widgets.HBox([w_ev_start, w_ev_end, w_ev_btn]))
display(out_evolution)
_on_update_evolution(None)

In [ ]:
# =============================================================================
# CELL 10 — US Curve-Regime Event Study on Soberanos
# =============================================================================
# Studies Soberano behavior during UST curve regime episodes.
# Product: [CMT Tenor, Spread to UST]
# Type: [Level, Spread, Butterfly] with tenor input boxes
# US Curve: which UST spread drives the regime (2s10s, etc.)
# Lookback: period for UST regime classification (5d or 20d)
# Regime: selected regime label
# Band: 1SD, 2SD, 3SD, 10/90pct, 25/75pct, None
# Red line: cumulative change since a specific regime start date (date picker)
# Gray lines: all historical paths for the selected regime
# Yellow line: mean path
# =============================================================================


def build_event_study_data(product, stype, t1, t2, t3,
                           ust_spread, ust_lookback,
                           regime_label, start_dt, end_dt, max_days=20):
    """Build all regime episode paths for the event study.
    
    Returns
    -------
    paths : list of arrays (cumulative bps changes, length up to max_days)
    path_dates : list of start dates for each path
    episodes_df : DataFrame of regime episodes
    current_regime : str — current regime label
    current_start : date — start of current regime
    current_duration : int — days in current regime
    """
    # 1) Get UST regime classification
    ust_pair = None
    for s, l in UST_SPREAD_PAIRS:
        if spread_label(s, l) == ust_spread:
            ust_pair = (s, l)
            break
    if ust_pair is None:
        return [], [], pd.DataFrame(), "", None, 0
    
    regime_df = classify_curve_regime(df_ust_bps, ust_pair[0], ust_pair[1], ust_lookback)
    
    # 2) Get Soberano series
    sobs_series, sobs_label = get_sobs_series(product, stype, t1, t2, t3)
    if sobs_series is None:
        return [], [], pd.DataFrame(), sobs_label, None, 0
    
    # Align dates
    common = regime_df.index.intersection(sobs_series.index)
    regime_df = regime_df.loc[common]
    sobs_aligned = sobs_series.loc[common]
    
    # Filter to date range
    mask = (regime_df.index >= pd.Timestamp(start_dt)) & (regime_df.index <= pd.Timestamp(end_dt))
    regime_df = regime_df.loc[mask]
    sobs_aligned = sobs_aligned.loc[mask]
    
    # 3) Extract episodes
    episodes = get_regime_episodes(regime_df["regime"])
    if episodes.empty:
        return [], [], episodes, "", None, 0
    
    # Current regime info
    current_regime = episodes.iloc[-1]["regime"]
    current_start = episodes.iloc[-1]["start"]
    current_duration = episodes.iloc[-1]["duration"]
    
    # 4) Filter episodes matching selected regime
    regime_episodes = episodes[episodes["regime"] == regime_label]
    
    # 5) Build paths: cumulative change in Soberano series from episode start
    paths = []
    path_start_dates = []
    for _, ep in regime_episodes.iterrows():
        ep_start = ep["start"]
        # Get index position of episode start
        try:
            start_idx = sobs_aligned.index.get_loc(ep_start)
        except KeyError:
            continue
        
        # Extract up to max_days from start
        end_idx = min(start_idx + max_days + 1, len(sobs_aligned))
        segment = sobs_aligned.iloc[start_idx:end_idx]
        
        if len(segment) < 2:
            continue
        
        # Cumulative change from day 0 (in bps)
        cum_chg = (segment.values - segment.values[0])
        paths.append(cum_chg)
        path_start_dates.append(ep_start)
    
    return paths, path_start_dates, regime_episodes, current_regime, current_start, current_duration


def plot_event_study(product, stype, t1, t2, t3,
                     ust_spread, ust_lookback, regime_label,
                     band, start_dt, end_dt, show_y_pct,
                     highlight_date=None):
    """Plot the full event study chart."""
    
    max_days = 20
    paths, path_dates, episodes_df, current_regime, current_start, current_dur = \
        build_event_study_data(product, stype, t1, t2, t3,
                               ust_spread, ust_lookback, regime_label,
                               start_dt, end_dt, max_days)
    
    _, sobs_label = get_sobs_series(product, stype, t1, t2, t3)
    
    if len(paths) == 0:
        print(f"No episodes found for regime '{regime_label}' in selected range")
        return
    
    # Stats for info box
    durations = episodes_df["duration"].values
    n_instances = len(episodes_df)
    longest = int(durations.max()) if len(durations) > 0 else 0
    shortest = int(durations.min()) if len(durations) > 0 else 0
    
    fig = go.Figure()
    
    # Pad paths to max_days+1 with NaN
    padded = np.full((len(paths), max_days + 1), np.nan)
    for i, p in enumerate(paths):
        padded[i, :len(p)] = p
    
    x_days = list(range(max_days + 1))
    
    # Compute mean and bands
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mean_path = np.nanmean(padded, axis=0)
        std_path = np.nanstd(padded, axis=0)
    
    # --- All historical paths (light gray, wider lines) ---
    for i in range(len(paths)):
        is_highlight = (highlight_date is not None and 
                       path_dates[i] == pd.Timestamp(highlight_date))
        if is_highlight:
            continue  # draw highlighted path separately
        
        fig.add_trace(go.Scatter(
            x=x_days[:len(paths[i])],
            y=paths[i],
            mode="lines",
            line=dict(color="rgba(180,180,180,0.35)", width=1.2),
            name="Historical paths" if i == 0 else None,
            showlegend=(i == 0),
            legendgroup="hist",
            hovertemplate=(f"Start: {path_dates[i].strftime('%Y-%m-%d')}<br>"
                          f"Day %{{x}}<br>Cum Δ: %{{y:.1f}} bps<extra></extra>"),
        ))
    
    # --- Mean path (yellow, thick) ---
    fig.add_trace(go.Scatter(
        x=x_days, y=mean_path,
        mode="lines",
        line=dict(color="#FFD600", width=3),
        name="Mean path",
        hovertemplate="Day %{x}<br>Mean Δ: %{y:.1f} bps<extra></extra>",
    ))
    
    # --- Bands ---
    if band != "None":
        if band in ["1SD", "2SD", "3SD"]:
            n_sd = int(band[0])
            upper = mean_path + n_sd * std_path
            lower = mean_path - n_sd * std_path
            band_label = f"±{n_sd}σ"
        elif band == "10/90 pct":
            upper = np.nanpercentile(padded, 90, axis=0)
            lower = np.nanpercentile(padded, 10, axis=0)
            band_label = "10th–90th pct"
        elif band == "25/75 pct":
            upper = np.nanpercentile(padded, 75, axis=0)
            lower = np.nanpercentile(padded, 25, axis=0)
            band_label = "25th–75th pct"
        else:
            upper = lower = None
            band_label = ""
        
        if upper is not None:
            fig.add_trace(go.Scatter(
                x=x_days, y=upper, mode="lines",
                line=dict(width=0), showlegend=False,
            ))
            fig.add_trace(go.Scatter(
                x=x_days, y=lower, mode="lines",
                line=dict(width=0), fill="tonexty",
                fillcolor="rgba(255,214,0,0.15)",
                name=band_label,
            ))
    
    # --- Highlighted path (red, thick) — specific episode date ---
    if highlight_date is not None:
        hl_ts = pd.Timestamp(highlight_date)
        for i, pd_date in enumerate(path_dates):
            if pd_date == hl_ts:
                fig.add_trace(go.Scatter(
                    x=x_days[:len(paths[i])],
                    y=paths[i],
                    mode="lines",
                    line=dict(color="#FF1744", width=3),
                    name=f"Path from {hl_ts.strftime('%Y-%m-%d')}",
                    hovertemplate=(f"Start: {hl_ts.strftime('%Y-%m-%d')}<br>"
                                  f"Day %{{x}}<br>Cum Δ: %{{y:.1f}} bps<extra></extra>"),
                ))
                break
    
    # --- Current regime path (red, if no highlight and current matches) ---
    elif current_regime == regime_label and len(path_dates) > 0:
        # Last path = current
        fig.add_trace(go.Scatter(
            x=x_days[:len(paths[-1])],
            y=paths[-1],
            mode="lines",
            line=dict(color="#FF1744", width=3),
            name=f"Current (from {path_dates[-1].strftime('%Y-%m-%d')})",
            hovertemplate=(f"Current episode<br>"
                          f"Day %{{x}}<br>Cum Δ: %{{y:.1f}} bps<extra></extra>"),
        ))
    
    # --- Info text ---
    regime_match = "matches current" if current_regime == regime_label else f"current is {current_regime}"
    
    title_text = (
        f"<b>Soberano Curve-Regime Event Study: {sobs_label}</b><br>"
        f"<span style='color:#00E5FF;font-size:11px'>"
        f"Current ({ust_spread}, {ust_lookback}d) as of {current_start.strftime('%Y-%m-%d') if current_start else 'N/A'}: {current_regime}</span><br>"
        f"<span style='color:#4CAF50;font-size:11px'>"
        f"In place for {current_dur} day(s) since {current_start.strftime('%Y-%m-%d') if current_start else 'N/A'}</span><br>"
        f"<span style='color:#FFD600;font-size:11px'>"
        f"Selected: {regime_label} ({regime_match})</span>"
    )
    
    stats_text = (f"Instances: {n_instances} | Longest: {longest}d | "
                  f"Shortest: {shortest}d | "
                  f"Range: {start_dt} to {end_dt}")
    
    fig.update_layout(
        height=550, width=1000,
        plot_bgcolor=BG_COLOR, paper_bgcolor=BG_COLOR,
        font=dict(color=TEXT_COLOR, size=10),
        title=dict(text=title_text, font=dict(size=13, color="#FFFFFF"), x=0.02, y=0.97),
        xaxis=dict(
            title="Days in Regime", gridcolor=GRID_COLOR,
            tickfont=dict(color=TEXT_COLOR), titlefont=dict(color=TEXT_COLOR),
            dtick=5,
        ),
        yaxis=dict(
            title="Cumulative Δ (bps)", gridcolor=GRID_COLOR, zerolinecolor="#555555",
            tickfont=dict(color=TEXT_COLOR), titlefont=dict(color=TEXT_COLOR),
        ),
        legend=dict(
            x=0.75, y=0.98, font=dict(size=9, color=TEXT_COLOR),
            bgcolor="rgba(0,0,0,0.6)", bordercolor=GRID_COLOR,
        ),
        margin=dict(l=60, r=30, t=110, b=50),
        annotations=[dict(
            text=stats_text, xref="paper", yref="paper",
            x=0.98, y=1.08, showarrow=False,
            font=dict(size=9, color="#888888"),
        )],
    )
    
    fig.show()


# ---------------------------------------------------------------------------
# Widget controls for Event Study
# ---------------------------------------------------------------------------
w_es_product = widgets.Dropdown(
    options=["CMT Tenor", "Spread to UST"], value="Spread to UST",
    description="Product:", layout=widgets.Layout(width="200px"))
w_es_type = widgets.Dropdown(
    options=["Level", "Spread", "Butterfly"], value="Level",
    description="Type:", layout=widgets.Layout(width="160px"))
w_es_t1 = widgets.Dropdown(options=[6,7,8,9,10,11,12,13,14,15], value=10, description="T1:")
w_es_t2 = widgets.Dropdown(options=[6,7,8,9,10,11,12,13,14,15], value=15, description="T2:")
w_es_t3 = widgets.Dropdown(options=[6,7,8,9,10,11,12,13,14,15], value=15, description="T3:")

w_es_ust_curve = widgets.Dropdown(
    options=["2s5s", "2s10s", "2s30s", "5s10s", "5s30s", "10s30s"],
    value="2s10s", description="US Curve:", layout=widgets.Layout(width="170px"))
w_es_lookback = widgets.Dropdown(
    options=[5, 20], value=20, description="Lookback:", layout=widgets.Layout(width="140px"))
w_es_regime = widgets.Dropdown(
    options=REGIME_LABELS, value="Bear Flattener",
    description="Regime:", layout=widgets.Layout(width="200px"))
w_es_band = widgets.Dropdown(
    options=["None", "1SD", "2SD", "3SD", "10/90 pct", "25/75 pct"],
    value="2SD", description="Band:", layout=widgets.Layout(width="160px"))

w_es_start = widgets.DatePicker(
    value=pd.Timestamp("2016-01-01"), description="Start:", layout=widgets.Layout(width="200px"))
w_es_end = widgets.DatePicker(
    value=df_ust.index.max().date(), description="End:", layout=widgets.Layout(width="200px"))

w_es_show_pct = widgets.Checkbox(value=False, description="Y as %", layout=widgets.Layout(width="100px"))

# Date picker for the highlighted red line (specific episode start date)
w_es_hl_toggle = widgets.Checkbox(value=False, description="Highlight specific episode",
                                   layout=widgets.Layout(width="220px"))
w_es_hl_date = widgets.DatePicker(
    value=df_ust.index.max().date(), description="Episode start:",
    layout=widgets.Layout(width="220px"))

w_es_btn = widgets.Button(description="Update", button_style="info")
out_event_study = widgets.Output()

def _update_es_type_vis(change):
    stype = w_es_type.value
    w_es_t2.layout.visibility = "visible" if stype in ["Spread", "Butterfly"] else "hidden"
    w_es_t3.layout.visibility = "visible" if stype == "Butterfly" else "hidden"

w_es_type.observe(_update_es_type_vis, names="value")
_update_es_type_vis(None)

def _update_hl_vis(change):
    w_es_hl_date.layout.visibility = "visible" if w_es_hl_toggle.value else "hidden"

w_es_hl_toggle.observe(_update_hl_vis, names="value")
_update_hl_vis(None)

def _on_update_es(b):
    with out_event_study:
        clear_output(wait=True)
        hl_date = w_es_hl_date.value if w_es_hl_toggle.value else None
        plot_event_study(
            product=w_es_product.value,
            stype=w_es_type.value,
            t1=w_es_t1.value,
            t2=w_es_t2.value,
            t3=w_es_t3.value,
            ust_spread=w_es_ust_curve.value,
            ust_lookback=w_es_lookback.value,
            regime_label=w_es_regime.value,
            band=w_es_band.value,
            start_dt=str(w_es_start.value),
            end_dt=str(w_es_end.value),
            show_y_pct=w_es_show_pct.value,
            highlight_date=hl_date,
        )

w_es_btn.on_click(_on_update_es)

display(widgets.HBox([w_es_product, w_es_type, w_es_t1, w_es_t2, w_es_t3]))
display(widgets.HBox([w_es_ust_curve, w_es_lookback, w_es_regime, w_es_band, w_es_show_pct]))
display(widgets.HBox([w_es_start, w_es_end, w_es_btn]))
display(widgets.HBox([w_es_hl_toggle, w_es_hl_date]))
display(out_event_study)
_on_update_es(None)